# Feature Engineering — Exercises

Companion to the note [Feature Engineering](Feature%20Engineering.md).

Practise encoding categorical, text and time features (one-hot, target encoding, hashing, TF-IDF, cyclic sin/cos), building polynomial and interaction terms, and selecting features with filter, wrapper and embedded methods.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from math import comb
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import f_regression
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mutual_info_score, roc_auc_score, r2_score
from sklearn.utils import murmurhash3_32
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Pick an encoding
*🧠 Concept · ★☆☆*

Choose an encoding for each feature and justify it, for (i) a linear model and (ii) a tree ensemble.

(a) `colour` ∈ {red, green, blue}. (b) `education` ∈ {primary, secondary, bachelor, master, PhD}. (c) `zip_code` with 30 000 levels.
(d) `timestamp` of a taxi ride. (e) a free-text product review. (f) `income`, heavily right-skewed.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is there an order? How many levels? Is there periodicity? Does the model care about monotone transforms?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) One-hot (no order) for both. (b) Ordinal integers preserve the order; for a linear model this assumes equal spacing, so one-hot is an alternative.
(c) Too many levels for one-hot: target encoding (out-of-fold!), hashing, or a learned embedding; trees handle target-encoded values well.
(d) Extract hour/day-of-week/month; cyclic sin/cos for the linear model, plain integers suffice for trees; plus holiday flags, lags.
(e) TF-IDF of words / n-grams, or sentence embeddings ([[Transformers]]). (f) $\log(1+x)$ or Box–Cox for the linear model; trees are invariant to monotone transforms.

</details>

### Exercise 2 · Why target encoding leaks
*🧠 Concept · ★★☆*

Target encoding replaces a category by the mean target of that category. (a) Explain why computing it on the full training set and then
training on it overfits, especially for rare categories. (b) Describe two fixes. (c) Why is this leakage hidden if you only look at training metrics?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A category seen once is encoded with exactly that sample's own label.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The encoding of a row contains **its own label**. For a category with one row, the feature equals $y$; the model learns to trust it,
but at test time new rows do not have this property.
(b) **Out-of-fold encoding** (encode each training row with statistics computed on the other folds) and **smoothing** towards the global
mean, $\tilde\mu_c = \frac{n_c\bar y_c + m\bar y}{n_c + m}$; adding noise also helps.
(c) Training (and in-sample CV after encoding) metrics look excellent; only a truly held-out set reveals the gap. See [[Data Preprocessing]].

</details>

### Exercise 3 · Filter, wrapper, embedded
*🧠 Concept · ★★☆*

Compare the three families of feature selection from the note on: computational cost, whether they account for feature interactions,
and whether they depend on a specific model. Give one example of each. Why must selection happen inside cross-validation?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A filter scores each feature alone; a wrapper retrains a model on feature subsets; embedded methods select while training.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | Filter | Wrapper | Embedded |
|---|---|---|---|
| Example | correlation, mutual information, F-test | RFE, forward selection | L1 / Lasso, tree importances |
| Cost | cheap ($O(d)$ univariate scores) | expensive (many refits) | one (or few) fits |
| Interactions | no (univariate) | yes | partly |
| Model-specific | no | yes | yes |

Selection uses the labels, so doing it on all data before CV leaks validation labels into the features and inflates scores
([[Cross-Validation and Model Selection]]).

</details>

### Exercise 4 · Cyclic features
*🧠 Concept · ★☆☆*

Why is `hour` ∈ {0,…,23} a poor numeric feature for a linear model, and why does $(\sin(2\pi h/24), \cos(2\pi h/24))$ fix it?
Why do you need **both** sine and cosine? Would a decision tree need this?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What is the distance between 23:00 and 00:00 in each representation? Is $\sin$ alone injective on a circle?

</details>

<details>
<summary><b>✅ Solution</b></summary>

As an integer, 23 and 0 are maximally far apart although they are adjacent, and a linear model can only express a monotone effect of the hour.
The sin/cos pair places hours on a circle, so neighbouring hours are close, and linear combinations $a\sin + b\cos$ express any shifted
daily sinusoid. Sine alone maps e.g. 3:00 and 9:00 to the same value. Trees can split hours into intervals and need it less, though
it can still save splits.

</details>

## Part B · By hand

### Exercise 5 · Distances on the clock
*✍️ By hand · ★☆☆*

Encode hour $h$ as $\phi(h) = (\sin(2\pi h/24), \cos(2\pi h/24))$. Compute the Euclidean distance $\lVert\phi(23) - \phi(1)\rVert$ and compare it with
$|23 - 1|$. Show that in general $\lVert\phi(a)-\phi(b)\rVert = 2\sin(\pi\,\Delta/24)$ with $\Delta$ the circular difference.

In [ ]:
dist_23_1 = None

_phi = lambda h: np.array([np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)])
check('distance', dist_23_1, np.linalg.norm(_phi(23) - _phi(1)))

<details>
<summary><b>💡 Hint</b></summary>

Two points on the unit circle separated by angle $\theta$ have chord length $2\sin(\theta/2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The angle between hours 23 and 1 is $2\pi\cdot2/24 = \pi/6$, the chord length is $2\sin(\pi/12)\approx0.518$, whereas the raw difference is 22
(and hours 11 and 13 are also at distance $0.518$). Chord: $\lVert\phi(a)-\phi(b)\rVert^2 = 2 - 2\cos\theta = 4\sin^2(\theta/2)$.

```python
dist_23_1 = 2 * np.sin(np.pi / 12)
```

</details>

### Exercise 6 · How many polynomial features?
*✍️ By hand · ★★☆*

`PolynomialFeatures(degree=p)` on $d$ inputs creates all monomials of total degree $\le p$ (including the constant).
Show that their number is $\binom{d+p}{p}$ and evaluate it for $d=10$, $p=2$ and $p=3$. What does this imply for $d=1000$?

In [ ]:
n_p2 = None
n_p3 = None

check('d=10, p=2', n_p2, PolynomialFeatures(2).fit(np.zeros((1, 10))).n_output_features_)
check('d=10, p=3', n_p3, PolynomialFeatures(3).fit(np.zeros((1, 10))).n_output_features_)

<details>
<summary><b>💡 Hint</b></summary>

Stars and bars: a monomial $x_1^{a_1}\cdots x_d^{a_d}$ with $\sum a_i\le p$ corresponds to $(a_0, a_1,\dots,a_d)$ with $a_0 = p - \sum a_i\ge0$ summing to exactly $p$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Non-negative integer solutions of $a_0 + a_1 + \dots + a_d = p$ number $\binom{p+d}{d} = \binom{d+p}{p}$.
$d=10$: $p=2$ gives $\binom{12}{2}=66$, $p=3$ gives $\binom{13}{3}=286$. For $d=1000,\ p=2$: $\binom{1002}{2} = 501\,501$ features.
Explicit expansion explodes, which motivates [[Kernel Methods]] (implicit feature maps) and regularization.

```python
n_p2 = comb(12, 2)
n_p3 = comb(13, 3)
```

</details>

### Exercise 7 · The dummy-variable trap
*✍️ By hand · ★☆☆*

A design matrix has an intercept column and a full one-hot encoding of a 3-level category, for 6 rows (two of each level).
What is its rank, and how many columns does it have? Why is $X^\top X$ singular, and what are two fixes?

In [ ]:
rank_X = None
n_cols = None

_cat = np.array([0,0,1,1,2,2])
_X = np.c_[np.ones(6), np.eye(3)[_cat]]
check('rank', rank_X, np.linalg.matrix_rank(_X)); check('columns', n_cols, _X.shape[1])

<details>
<summary><b>💡 Hint</b></summary>

The three one-hot columns sum to the intercept column.

</details>

<details>
<summary><b>✅ Solution</b></summary>

4 columns but rank 3: $\mathbf 1 = e_1 + e_2 + e_3$, an exact linear dependence, so $X^\top X$ is singular and OLS coefficients are not unique.
Fixes: drop one level (`OneHotEncoder(drop='first')`, the dropped level becomes the reference), or keep all levels and add a ridge penalty.

```python
rank_X = 3
n_cols = 4
```

</details>

### Exercise 8 · TF-IDF by hand
*✍️ By hand · ★★☆*

Corpus: $d_1$ = "cat sat mat", $d_2$ = "cat cat dog", $d_3$ = "dog barked". With raw term counts as tf and sklearn's smoothed
$\text{idf}(t) = \ln\frac{1+n}{1+\text{df}(t)} + 1$, compute the (un-normalised) tf-idf of "cat" in $d_2$ and of "barked" in $d_3$.

In [ ]:
tfidf_cat_d2 = None
tfidf_barked_d3 = None

_v = TfidfVectorizer(norm=None); _M = _v.fit_transform(['cat sat mat', 'cat cat dog', 'dog barked']).toarray()
_voc = _v.vocabulary_
check('cat in d2', tfidf_cat_d2, _M[1, _voc['cat']]); check('barked in d3', tfidf_barked_d3, _M[2, _voc['barked']])

<details>
<summary><b>💡 Hint</b></summary>

$n=3$. df(cat) $=2$, df(barked) $=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

idf(cat) $= \ln(4/3)+1 \approx 1.288$, tf $=2$ → $2.575$. idf(barked) $= \ln(4/2)+1\approx1.693$, tf $=1$ → $1.693$.
Rare words get larger weights; words occurring in every document get the minimum idf of 1. sklearn then L2-normalises each row by default.

```python
tfidf_cat_d2 = 2 * (np.log(4/3) + 1)
tfidf_barked_d3 = np.log(4/2) + 1
```

</details>

### Exercise 9 · Mutual information by hand
*✍️ By hand · ★★☆*

A binary feature $X$ and label $Y$ have joint distribution $p(0,0)=0.4,\ p(0,1)=0.1,\ p(1,0)=0.1,\ p(1,1)=0.4$.
Compute $I(X;Y) = \sum_{x,y} p(x,y)\ln\frac{p(x,y)}{p(x)p(y)}$ in nats. What is $I$ if $X$ and $Y$ are independent?

In [ ]:
mi_hand = None

check('mutual information', mi_hand, mutual_info_score(None, None, contingency=np.array([[40, 10], [10, 40]])), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Both marginals are $(0.5, 0.5)$, so $p(x)p(y) = 0.25$ in every cell.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$I = 2\cdot0.4\ln\frac{0.4}{0.25} + 2\cdot0.1\ln\frac{0.1}{0.25} = 0.8\ln1.6 + 0.2\ln0.4 = 0.3760 - 0.1833 = 0.1927$ nats.
Independent: every log term is $\ln1 = 0$, so $I=0$. Unlike correlation, MI also detects non-linear dependence, which makes it a good filter score.

```python
mi_hand = 0.8 * np.log(1.6) + 0.2 * np.log(0.4)
```

</details>

### Exercise 10 · Smoothed target encoding
*✍️ By hand · ★☆☆*

The global positive rate is $\bar y = 0.2$. Category A has 4 rows with 3 positives, category B has 400 rows with 120 positives.
Compute the smoothed encodings $\tilde\mu_c = \frac{n_c\bar y_c + m\bar y}{n_c + m}$ with $m = 10$. Which one is pulled more toward the prior?

In [ ]:
enc_A = None
enc_B = None

def _enc(pos, n, m=10, prior=0.2):
    # posterior mean of a Beta(m*prior, m*(1-prior)) prior after observing pos / n
    a, b = m * prior + pos, m * (1 - prior) + n - pos
    return a / (a + b)
check('A', enc_A, _enc(3, 4)); check('B', enc_B, _enc(120, 400))

<details>
<summary><b>💡 Hint</b></summary>

A: $(3 + 10\cdot0.2)/(4 + 10)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A: $(3 + 2)/14 \approx 0.357$ (raw mean $0.75$, strongly shrunk). B: $(120 + 2)/410 \approx 0.298$ (raw $0.30$, barely changed).
This is the posterior mean under a Beta prior with $m$ pseudo-observations: rare categories are not trusted.

```python
enc_A = 5 / 14
enc_B = 122 / 410
```

</details>

## Part C · Code

### Exercise 11 · One-hot encoding from scratch
*💻 Code · ★☆☆*

Implement `fit_onehot(col)` returning the sorted categories and `transform_onehot(col, cats)` returning a 0/1 matrix, where unseen categories
become an all-zero row (like `handle_unknown='ignore'`).

In [ ]:
train_col = np.array(['red', 'green', 'blue', 'green', 'red'])
test_col = np.array(['blue', 'purple', 'red'])

def fit_onehot(col):
    return None

def transform_onehot(col, cats):
    return None

oh_test = transform_onehot(test_col, fit_onehot(train_col))

check('one-hot', oh_test, OneHotEncoder(handle_unknown='ignore').fit(train_col[:, None]).transform(test_col[:, None]).toarray())

<details>
<summary><b>💡 Hint</b></summary>

`np.unique` returns sorted categories; `(col[:, None] == cats[None, :]).astype(float)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Broadcasting the comparison builds the whole matrix at once. Categories must be learned on train only; unseen test categories need an explicit policy.

```python
train_col = np.array(['red', 'green', 'blue', 'green', 'red'])
test_col = np.array(['blue', 'purple', 'red'])

def fit_onehot(col):
    return np.unique(col)

def transform_onehot(col, cats):
    return (col[:, None] == cats[None, :]).astype(float)

oh_test = transform_onehot(test_col, fit_onehot(train_col))
```

</details>

### Exercise 12 · Interaction terms solve XOR
*💻 Code · ★★☆*

For $x\sim U(-1,1)^2$ and label $y = [x_1x_2 > 0]$ (XOR pattern), compare logistic regression on $(x_1,x_2)$ (`acc_plain`) with logistic
regression on $(x_1, x_2, x_1x_2)$ (`acc_inter`), both on a held-out split.

In [ ]:
X_x = rng.uniform(-1, 1, size=(600, 2))
y_x = (X_x[:, 0] * X_x[:, 1] > 0).astype(int)
acc_plain = None
acc_inter = None

if acc_plain is not None:
    check('plain ~ chance', acc_plain < 0.65, True)
    check('interaction > 0.95', acc_inter > 0.95, True)
else:
    check('XOR', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.c_[X_x, X_x[:, 0] * X_x[:, 1]]`. Use `train_test_split(..., random_state=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

No line separates XOR quadrants, so the plain model is at chance (≈ 0.5). With $\phi(x) = (x_1, x_2, x_1x_2)$ the classes are separated by the
hyperplane $x_1x_2 = 0$ and accuracy is ≈ 0.97: a non-linear feature map turns a linear model non-linear ([[Linear Models for Classification]]).

```python
X_x = rng.uniform(-1, 1, size=(600, 2))
y_x = (X_x[:, 0] * X_x[:, 1] > 0).astype(int)
def _acc(F):
    a, b, c, d = train_test_split(F, y_x, random_state=0)
    return LogisticRegression(C=100).fit(a, c).score(b, d)
acc_plain = _acc(X_x)
acc_inter = _acc(np.c_[X_x, X_x[:, 0] * X_x[:, 1]])
print(acc_plain, acc_inter)
```

</details>

### Exercise 13 · Cyclic hour features
*💻 Code · ★★☆*

Simulated bike demand peaks in the late afternoon: `demand = 50 + 30*cos(2π(h-17)/24) + noise`. Fit linear regression with (a) the raw hour
and (b) the sin/cos encoding. Store test $R^2$ in `r2_raw`, `r2_cyc`.

In [ ]:
hours = rng.integers(0, 24, 1000)
demand = 50 + 30 * np.cos(2 * np.pi * (hours - 17) / 24) + 5 * rng.normal(size=1000)

def cyclic(h, period=24):
    return None   # (N, 2) array

r2_raw = None
r2_cyc = None

check('cyclic encoding', cyclic(np.array([0, 6])), [[0, 1], [1, 0]])
if r2_raw is not None:
    check('cyclic R^2 > 0.9 and far above raw', r2_cyc > 0.9 and r2_cyc > r2_raw + 0.3, True)
else:
    check('R^2 comparison', None, True)

<details>
<summary><b>💡 Hint</b></summary>

$\cos(\theta - \theta_0) = \cos\theta_0\cos\theta + \sin\theta_0\sin\theta$, so a linear model on (sin, cos) can represent the truth exactly.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Raw hour: $R^2\approx0.5$ (a straight line through a daily cycle only captures the rising part). Sin/cos: $R^2\approx0.95$, the noise ceiling, since the true function is linear
in these two features.

```python
hours = rng.integers(0, 24, 1000)
demand = 50 + 30 * np.cos(2 * np.pi * (hours - 17) / 24) + 5 * rng.normal(size=1000)

def cyclic(h, period=24):
    a = 2 * np.pi * np.asarray(h) / period
    return np.c_[np.sin(a), np.cos(a)]

def _r2(F):
    a, b, c, d = train_test_split(F, demand, random_state=0)
    return r2_score(d, LinearRegression().fit(a, c).predict(b))
r2_raw = _r2(hours[:, None].astype(float))
r2_cyc = _r2(cyclic(hours))
print(r2_raw, r2_cyc)
```

</details>

### Exercise 14 · Out-of-fold target encoding
*💻 Code · ★★★*

`city` has 400 levels and **no** relation to the label. Implement
`target_encode_oof(cat, y, n_splits=5, m=0)`: each row is encoded with the mean target of its category computed on the **other** folds
(fall back to the global mean of those folds for unseen categories). Compare a logistic regression trained on (a) naive full-train encodings and
(b) OOF encodings: report train and test AUC for both.

In [ ]:
n = 3000
city = rng.integers(0, 400, n)
x_real = rng.normal(size=n)
y_te_demo = (x_real + rng.normal(size=n) > 0).astype(int)     # city is pure noise
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.5, random_state=0)

def naive_encode(cat_tr, y_tr, cat_apply):
    means = {c: y_tr[cat_tr == c].mean() for c in np.unique(cat_tr)}
    return np.array([means.get(c, y_tr.mean()) for c in cat_apply])

def target_encode_oof(cat, y, n_splits=5):
    return None

enc_oof = target_encode_oof(city[idx_tr], y_te_demo[idx_tr])
auc = None   # dict: {'naive_train':..., 'naive_test':..., 'oof_train':..., 'oof_test':...}

if enc_oof is not None:
    _kf = KFold(5, shuffle=True, random_state=0); _c, _yv = city[idx_tr], y_te_demo[idx_tr]
    _ref = np.empty(len(_c))
    for a, b in _kf.split(_c):
        _ref[b] = naive_encode(_c[a], _yv[a], _c[b])
    check('OOF encoding (KFold(5, shuffle=True, random_state=0))', enc_oof, _ref)
    check('naive encoding: big train/test gap', auc['naive_train'] - auc['naive_test'] > 0.05, True)
    check('OOF: train AUC is honest', abs(auc['oof_train'] - auc['oof_test']) < 0.03, True)
else:
    check('OOF encoding', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Use `KFold(5, shuffle=True, random_state=0)` and reuse `naive_encode(cat[train_folds], y[train_folds], cat[held_out_fold])`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Test rows are always encoded with the full-training-set means (`naive_encode(city[idx_tr], y[idx_tr], city[idx_te])`).

</details>

<details>
<summary><b>✅ Solution</b></summary>

With naive encoding each city's value contains the rows' own labels (≈ 7 rows per city), so on the training set the noise feature looks
predictive and the model puts weight on it: train AUC ≈ 0.89 but test AUC ≈ 0.73. With OOF encoding the training rows see the
feature as it will look on new data (noisy), the model ignores it, and train and test AUC agree (≈ 0.83), which is also a *better* test AUC than the naive model.

```python
n = 3000
city = rng.integers(0, 400, n)
x_real = rng.normal(size=n)
y_te_demo = (x_real + rng.normal(size=n) > 0).astype(int)
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.5, random_state=0)

def naive_encode(cat_tr, y_tr, cat_apply):
    means = {c: y_tr[cat_tr == c].mean() for c in np.unique(cat_tr)}
    return np.array([means.get(c, y_tr.mean()) for c in cat_apply])

def target_encode_oof(cat, y, n_splits=5):
    out = np.empty(len(cat))
    for a, b in KFold(n_splits, shuffle=True, random_state=0).split(cat):
        out[b] = naive_encode(cat[a], y[a], cat[b])
    return out

enc_oof = target_encode_oof(city[idx_tr], y_te_demo[idx_tr])
y_tr_, y_te_ = y_te_demo[idx_tr], y_te_demo[idx_te]
enc_te = naive_encode(city[idx_tr], y_tr_, city[idx_te])
auc = {}
for name, enc_tr in [('naive', naive_encode(city[idx_tr], y_tr_, city[idx_tr])), ('oof', enc_oof)]:
    F_tr, F_te = np.c_[x_real[idx_tr], enc_tr], np.c_[x_real[idx_te], enc_te]
    m = LogisticRegression().fit(F_tr, y_tr_)
    auc[name + '_train'] = roc_auc_score(y_tr_, m.predict_proba(F_tr)[:, 1])
    auc[name + '_test'] = roc_auc_score(y_te_, m.predict_proba(F_te)[:, 1])
print({k: round(v, 3) for k, v in auc.items()})
```

</details>

### Exercise 15 · The hashing trick
*💻 Code · ★★☆*

Implement `hash_features(tokens_list, n_features)` like sklearn's `FeatureHasher(input_type='string')`: for each token compute
`h = murmurhash3_32(token, positive=False)`, add `+1` if `h >= 0` else `-1` at column `abs(h) % n_features`. Why the random sign?

In [ ]:
docs_tok = [['cat', 'sat', 'mat'], ['cat', 'cat', 'dog'], ['zip_10115', 'zip_80331', 'dog']]

def hash_features(tokens_list, n_features=16):
    return None

H = hash_features(docs_tok, 16)

check('matches FeatureHasher', H, FeatureHasher(16, input_type='string').transform(docs_tok).toarray())

<details>
<summary><b>💡 Hint</b></summary>

Loop over documents and tokens; start from `np.zeros((len(tokens_list), n_features))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Hashing maps an unbounded vocabulary (e.g. 30 000 zip codes, unseen words) into a fixed number of columns without storing a vocabulary.
Collisions are unavoidable; the random sign makes colliding contributions cancel **in expectation**, so inner products between hashed vectors
are unbiased estimates of the original ones.

```python
docs_tok = [['cat', 'sat', 'mat'], ['cat', 'cat', 'dog'], ['zip_10115', 'zip_80331', 'dog']]

def hash_features(tokens_list, n_features=16):
    H = np.zeros((len(tokens_list), n_features))
    for i, toks in enumerate(tokens_list):
        for t in toks:
            h = murmurhash3_32(t, positive=False)
            H[i, abs(h) % n_features] += 1 if h >= 0 else -1
    return H

H = hash_features(docs_tok, 16)
```

</details>

### Exercise 16 · TF-IDF from scratch
*💻 Code · ★★☆*

Implement `tfidf(docs)` returning `(vocab, M)` with `vocab` sorted, raw counts as tf, smoothed idf $\ln\frac{1+n}{1+\text{df}}+1$, and
L2-normalised rows. Compare with `TfidfVectorizer()` (default settings) on the corpus below.

In [ ]:
corpus = ['the cat sat on the mat', 'the dog sat', 'cats and dogs', 'the mat was red']

def tfidf(docs):
    return None

res_tfidf = tfidf(corpus)

_tv = TfidfVectorizer().fit(corpus)
check('vocabulary', None if res_tfidf is None else list(res_tfidf[0]) == list(_tv.get_feature_names_out()), True)
check('matrix', None if res_tfidf is None else res_tfidf[1], _tv.transform(corpus).toarray())

<details>
<summary><b>💡 Hint</b></summary>

Tokenise with `doc.split()` (all words here have ≥ 2 characters, matching sklearn's default token pattern).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bag-of-words loses word order; TF-IDF down-weights ubiquitous words like "the". Note that "cat"/"cats" and "dog"/"dogs" are different
features: stemming or embeddings would merge them.

```python
corpus = ['the cat sat on the mat', 'the dog sat', 'cats and dogs', 'the mat was red']

def tfidf(docs):
    toks = [d.lower().split() for d in docs]
    vocab = sorted(set(t for d in toks for t in d))
    ix = {w: j for j, w in enumerate(vocab)}
    tf = np.zeros((len(docs), len(vocab)))
    for i, d in enumerate(toks):
        for t in d:
            tf[i, ix[t]] += 1
    df = (tf > 0).sum(0)
    M = tf * (np.log((1 + len(docs)) / (1 + df)) + 1)
    return vocab, M / np.linalg.norm(M, axis=1, keepdims=True)

res_tfidf = tfidf(corpus)
```

</details>

### Exercise 17 · Filter vs. embedded selection
*💻 Code · ★★☆*

50 features, only the first 5 matter. (a) Filter: implement `corr_filter(X, y, k)` returning the indices of the $k$ features with the largest
$|\text{corr}(x_j, y)|$. (b) Embedded: fit `Lasso(alpha=0.1)` and store the indices of non-zero coefficients in `lasso_sel`.
Do both recover {0,…,4}?

In [ ]:
X_fs = rng.normal(size=(300, 50))
y_fs = X_fs[:, :5] @ np.array([3, -2, 2, 1.5, -1]) + rng.normal(size=300)

def corr_filter(X, y, k):
    return None

filt_sel = corr_filter(X_fs, y_fs, 5)
lasso_sel = None

check('filter = top-k by F-statistic', None if filt_sel is None else sorted(filt_sel), sorted(np.argsort(-f_regression(X_fs, y_fs)[0])[:5]))
check('lasso finds the informative features', None if lasso_sel is None else set(range(5)) <= set(np.ravel(lasso_sel).tolist()), True)

<details>
<summary><b>💡 Hint</b></summary>

Standardise: $\text{corr} = \frac{1}{N}\sum z_{x}z_{y}$. The univariate F-statistic is a monotone function of $|\text{corr}|$, so the rankings agree.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both find the 5 informative features here. Lasso may keep a few noise features with tiny coefficients; a filter could miss a feature that only
matters jointly with others (e.g. an XOR interaction), which an embedded or wrapper method can catch.

```python
X_fs = rng.normal(size=(300, 50))
y_fs = X_fs[:, :5] @ np.array([3, -2, 2, 1.5, -1]) + rng.normal(size=300)

def corr_filter(X, y, k):
    Zx = (X - X.mean(0)) / X.std(0)
    zy = (y - y.mean()) / y.std()
    r = Zx.T @ zy / len(y)
    return np.argsort(-np.abs(r))[:k]

filt_sel = corr_filter(X_fs, y_fs, 5)
lasso_sel = np.flatnonzero(Lasso(alpha=0.1).fit(X_fs, y_fs).coef_)
print(sorted(filt_sel), lasso_sel)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Feature Engineering](Feature%20Engineering.md).*